# Labelled corpora

Counts for the Methods table. Nothing is written; the loaders fetch on demand.

In [1]:
import sys

sys.path.insert(0, "..")
import polars as pl

from config import INT2NAME, SHAH_SEEDS
from data.loader_twd_labelled import fetch_splits, load_splits
from data.loader_wcb_labelled import fetch_annotated

## TWD — benchmark splits

In [2]:
for seed in SHAH_SEEDS:
    load_splits("benchmark", seed=seed)

twd = fetch_splits(verbose=False).filter(pl.col("seed") == SHAH_SEEDS[0])
print("rows:", len(twd), "| unique sentences:", twd["sentence"].n_unique())
print("years:", twd["year"].min(), "->", twd["year"].max())
print("labels:", {INT2NAME[r["label"]]: r["count"] for r in twd["label"].value_counts().sort("label").to_dicts()})

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


rows: 2480 | unique sentences: 2419
years: 1996 -> 2022
labels: {'dovish': 650, 'hawkish': 606, 'neutral': 1224}


## WCB — 24-bank annotated sentences

In [3]:
wcb = fetch_annotated()
print("banks:", wcb["bank_name"].nunique())
print("labels:", wcb["label"].value_counts().to_dict())

raw rows: 25000 | stance labels: {'neutral': 8737, 'dovish': 8312, 'hawkish': 7097, 'irrelevant': 854}
after dropping irrelevant, fomc and duplicates: 23182
banks: 24
labels: {'neutral': 8404, 'dovish': 7943, 'hawkish': 6835}


## Summary

In [4]:
twd_pct = (twd["label"].value_counts(normalize=True).sort("label")["proportion"] * 100).round(0).to_list()
print("twd:", len(twd), "|", {INT2NAME[i]: p for i, p in enumerate(twd_pct)})
print("wcb:", len(wcb), "|", (wcb["label"].value_counts(normalize=True) * 100).round(0).to_dict())

twd: 2480 | {'dovish': 26.0, 'hawkish': 24.0, 'neutral': 49.0}
wcb: 23182 | {'neutral': 36.0, 'dovish': 34.0, 'hawkish': 29.0}
